# Stage 5 — Unsloth GRPO RL training (Colab, A100 80 GB)

Trains **one** arena policy at a time with GRPO on judge-scored rollouts,
starting from the Stage 4 **SFT v3 adapters**:

- attacker: `hf-user/Qwen3.8-27B-SFT-Attacker-v3`
- defender: `hf-user/Qwen3.8-27B-SFT-Defender-v3`

> **One policy per Colab session (attacker XOR defender).** A 27B QLoRA run
> needs the whole A100 80 GB; training both in one session OOMs. Set `AGENT`
> in section 1, run the notebook, then **restart the runtime**
> (`Runtime > Restart session`) and re-run with the other `AGENT`. The
> `rl.colab.yaml` overlay turns on a guard that refuses the second policy
> without a restart.

Both adapters are r=64 LoRA on `unsloth/Qwen3.8-27B-unsloth-bnb-4bit`; Stage 5
loads that base, rebuilds the identical adapter recipe and copies the SFT
weights in (`model.sft_adapters`).

This notebook applies `configs/rl.colab.yaml` on top of
`configs/rl.default.yaml` (`run.colab: true`, `adamw_8bit`, seq 16384 /
completion 8192) and consumes the arena's offline rollout JSONL files in
`qwen_dataset/datasets/rl_rounds/round_XX/<agent>.jsonl`. PLAN §7 constraints
stay baked in: lr 1e-4 (AGENT.md says 4e-4; PLAN supersedes), no warmup,
trained != initial, and attacker adapters != defender adapters (`verify.py`,
compared across the two separate sessions).

On AMD Instinct MI300X (DigitalOcean / AMD Developer Cloud) skip the CUDA
install cell and run `bash RL_pentest_network/scripts/setup_amd_mi300x.sh`
(`make rl-amd-setup`); drop the `rl.colab.yaml` overlay there so both policies
can train on the 192 GB card. Split topology: collect rollouts locally with
`--rollouts-only`, ship them with `make rl-sync RL_REMOTE=user@host`, then run
this notebook on the GPU host in offline mode.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"{torch.cuda.device_count()}x {props.name} {props.total_memory / 1024**3:.1f} GB")
    if props.total_memory / 1024**3 < 70:
        print("WARNING: Stage 5 Colab is tuned for an A100 80 GB; on smaller "
              "GPUs use --smoke and lower model.max_seq_length/completion.")

## 1. Pick ONE policy for this Colab session

Attacker XOR defender — never both. To switch policies later, restart the
runtime (`Runtime > Restart session`) first: the trainer's Colab guard refuses
to load a second policy in the same runtime.

In [ ]:
import os, uuid

# ONE policy per Colab session. To train the other policy afterwards:
#   Runtime > Restart session  ->  change AGENT  ->  run all cells again.
AGENT = os.environ.get("RL_AGENT", "attacker")   # "attacker" | "defender"
ROUND = int(os.environ.get("RL_ROUND", "0"))     # round_XX dataset to train on
assert AGENT in ("attacker", "defender"), f"bad AGENT {AGENT!r}"

# A100 80 GB overlay: run.colab=true (single-policy guard) + Colab tuning.
os.environ["RL_CONFIG_OVERLAY"] = "RL_pentest_network/configs/rl.colab.yaml"
# Stable id for this runtime; the CLI refuses a different policy under the same
# id, so separate `!python` subprocesses cannot bypass the guard.
os.environ["RL_COLAB_SESSION"] = os.environ.get("RL_COLAB_SESSION") or uuid.uuid4().hex
print(f"policy: {AGENT} | round: {ROUND} | session: {os.environ['RL_COLAB_SESSION']}")
print("to switch policy: Runtime > Restart session, then change AGENT")

## 2. Install the pinned stack

Following the Unsloth Qwen3.8 guide + GRPO template (2026-09): `unsloth` first
(track main for the qwen3_5 GRPO kernels), `trl==1.9.2`, `transformers==5.15.1`,
`datasets==4.3.0`. `rl/train.py` imports `unsloth` before `trl` itself.

In [ ]:
%%capture
!pip install -q --upgrade uv
!uv pip install -qqq torch triton bitsandbytes "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install -qqq "transformers==5.15.1"
# Unsloth main carries the qwen3_5 GRPO log-softmax fixes (same stack as the
# Qwen3.8/Muse GRPO templates); rl.grpo_config drops kwargs (e.g.
# max_prompt_length) removed by TRL 1.9 automatically.
!uv pip install -qqq --no-deps "tokenizers>=0.22.0,<=0.23.0" "trl==1.9.2" \
    "unsloth @ git+https://github.com/unslothai/unsloth" \
    "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo"
!uv pip install -qqq "peft>=0.20" pyyaml "torchao>=0.16.0"

## 3. Repo + rollout data

Clone the repo (or set `RL_REPO`), mount Drive and copy the arena rollout
JSONL files into `qwen_dataset/datasets/rl_rounds/round_XX/<agent>.jsonl`
(the `rollouts.dataset_dir` layout the colab overlay reads). Both the arena
log layout (`rl_rounds/r<N>/rollouts/<agent>.jsonl`) and
(`rl_rounds/round_XX/<agent>.jsonl`) are accepted.

In [ ]:
import os, subprocess
REPO = os.environ.get("RL_REPO", "/content/RL_cyberbench")
REPO_URL = os.environ.get("RL_REPO_URL", "")
if not os.path.isdir(os.path.join(REPO, "RL_pentest_network")) and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
assert os.path.isdir(os.path.join(REPO, "RL_pentest_network")), "set RL_REPO/RL_REPO_URL"
os.chdir(REPO)
%env PYTHONPATH={REPO}/RL_pentest_network:{REPO}/llm_pentest_network
print("repo:", REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = os.environ.get("RL_DATA", "/content/drive/MyDrive/pentest_rl")

import glob, os, re, shutil

def _round_no(path: str):
    for part in reversed(path.split(os.sep)):
        match = re.search(r"^(?:round_|r)(\d+)$", part)
        if match:
            return int(match.group(1))
    return None

# Normalize whatever Drive layout into dataset_dir's round_XX/<agent>.jsonl.
dest_root = os.path.join(REPO, "qwen_dataset", "datasets", "rl_rounds")
for src in sorted(glob.glob(os.path.join(DATA, "rl_rounds", "**", "*.jsonl"), recursive=True)):
    agent = os.path.splitext(os.path.basename(src))[0]
    round_no = _round_no(os.path.relpath(src, DATA))
    if agent not in ("attacker", "defender") or round_no is None:
        print("skip (path needs r<N>/round_XX and <agent>.jsonl):", src)
        continue
    dest = os.path.join(dest_root, f"round_{round_no:02d}", f"{agent}.jsonl")
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy(src, dest)
    print("copied", src, "->", os.path.relpath(dest, REPO))
for path in sorted(glob.glob(os.path.join(dest_root, "round_*", "*.jsonl"))):
    print(path)

## 4. Preflight (dry run)

Resolves the base model, the per-agent SFT v3 adapter, the LoRA shape, batch
algebra and exports for **this one policy** without loading weights. The
`rl.colab.yaml` overlay is picked up from `RL_CONFIG_OVERLAY`.

In [ ]:
!python -m rl.train --config RL_pentest_network/configs/rl.default.yaml --round {ROUND} --agent {AGENT} --dry-run

## 5. Smoke run (optional)

Four rollouts / one optimizer step on a reduced batch to validate the stack
before the real run.

In [ ]:
!python -m rl.train --config RL_pentest_network/configs/rl.default.yaml --round {ROUND} --agent {AGENT} --smoke

## 6. Train ONLY this policy

The run starts from `AGENT`'s Stage 4 adapter (`model.sft_adapters`) on top of
`unsloth/Qwen3.8-27B-unsloth-bnb-4bit`; `run.json` records it under
`sft_adapter` / `lora_init.init_from`.

Do **not** train the other policy in this session: restart the runtime first.
The Colab guard (`run.colab: true`) refuses a second policy anyway, for both
the in-process API and (via `RL_COLAB_SESSION`) separate `!python` cells.

In [ ]:
!python -m rl.train --config RL_pentest_network/configs/rl.default.yaml --round {ROUND} --agent {AGENT}

## 7. Verify (trained != initial; compare only across sessions)

In [ ]:
!python -m rl.verify --all --output-root rl_runs
other = "defender" if AGENT == "attacker" else "attacker"
if os.path.isdir(f"rl_runs/round_{ROUND:02d}/{other}/adapters"):
    !python -m rl.verify --compare rl_runs/round_{ROUND:02d}/attacker/adapters rl_runs/round_{ROUND:02d}/defender/adapters
else:
    print(f"no {other} adapters here yet: train {other} in a fresh runtime, "
          "then re-run this cell to compare attacker vs defender")

## 8. Collect artifacts

Adapters + manifests are small enough for Drive. GGUFs are ~15-16 GB each:
copy them to Drive, or promote locally with `python -m rl.export`-driven
`export.promote: true` (see README).

In [ ]:
import glob, os, shutil, tarfile
os.makedirs(DATA, exist_ok=True)
for pattern in ("rl_runs/*/*/adapters", "rl_runs/*/*/run.json", "rl_runs/*/*/verify.json", "rl_runs/*/*/weights_*.json"):
    for path in glob.glob(pattern):
        dest = os.path.join(DATA, path)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.isdir(path):
            shutil.copytree(path, dest, dirs_exist_ok=True)
        else:
            shutil.copy(path, dest)
for gguf in glob.glob("rl_runs/*/*/gguf/**/*.gguf", recursive=True):
    print("GGUF:", gguf, "-> copy to Drive / promote locally")
archive = "/content/rl_adapters.tgz"
with tarfile.open(archive, "w:gz") as tar:
    tar.add("rl_runs")
print("archive:", archive)

## 9. Arena promotion + switch-policy checklist

```bash
# before the first LIVE round: seed the arena with the SFT v3 starting policy
python -m rl.export --from-adapter --agent attacker --seed-volume --restart-sidecar
python -m rl.export --from-adapter --agent defender --seed-volume --restart-sidecar
# after each round: GGUF Q4_K_M -> models/rl + model volume + llama-server restart
python -m rl.export ...   # or set export.promote/seed_volume/restart_sidecar: true
# next RL round can warm-start from the previous round's adapters:
#   lora.init_from: rl_runs/round_00/attacker/adapters
```

Switching policies on Colab:

1. Download/copy this session's adapters + GGUF to Drive (section 8).
2. `Runtime > Restart session` (disconnect/delete the runtime is safest).
3. Re-run the notebook with the other `AGENT`.

Known risks: lr 1e-4 is above Unsloth's 5e-6 GRPO guidance (spec value);
`fast_inference=False` because vLLM cannot serve `qwen3_5` yet.
Keep the exported GGUF aligned with the SFT/GRPO base
(`unsloth/Qwen3.8-27B-unsloth-bnb-4bit`), not the old abliterated GGUF.